# Interactive ChlorA Heatmap — CalCOFI
Trains a model (RF vs Gradient Boosting) to predict Chlorophyll-A from location-independent oceanographic features, then renders an interactive heatmap over the California coast with sliders for the most important variables.

In [16]:

%pip install scikit-learn plotly ipywidgets pandas numpy anywidget

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('../Data/Finalized_CalCOFI_Phytoplankton.csv')

# Location-independent candidate features only (exclude Shore_km — derived from lat/lon)
CANDIDATE_FEATURES = [
    'T_degC', 'Salnty', 'O2ml_L', 'O2Sat',
    'PO4uM', 'SiO3uM', 'NO3uM', 'Cloud_Amt', 'NP_Ratio', 'Phaeop'
]
TARGET = 'ChlorA'

df_clean = df[CANDIDATE_FEATURES + [TARGET, 'Lat_Dec', 'Lon_Dec']].dropna()
X_all = df_clean[CANDIDATE_FEATURES]
y     = df_clean[TARGET]

print(f"Samples after dropping NaN: {len(df_clean):,}")
print(f"ChlorA range: {y.min():.3f} – {y.max():.3f} μg/L  |  mean: {y.mean():.3f}")

Samples after dropping NaN: 32,452
ChlorA range: 0.000 – 31.280 μg/L  |  mean: 0.906


In [2]:
# ── Compare Random Forest vs Gradient Boosting (5-fold CV) ───────────────────
print("Evaluating Random Forest (5-fold CV)...")
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_rmse = -cross_val_score(rf, X_all, y, cv=5,
                            scoring='neg_root_mean_squared_error',
                            n_jobs=-1).mean()

print("Evaluating Gradient Boosting (5-fold CV)...")
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=5, random_state=42)
gb_rmse = -cross_val_score(gb, X_all, y, cv=5,
                            scoring='neg_root_mean_squared_error').mean()

print(f"\nRandom Forest  CV-RMSE: {rf_rmse:.4f}")
print(f"Gradient Boost CV-RMSE: {gb_rmse:.4f}")

# Pick the model with lower RMSE and fit on full data
if rf_rmse <= gb_rmse:
    best_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    model_name = "Random Forest"
else:
    best_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                            max_depth=5, random_state=42)
    model_name = "Gradient Boosting"

print(f"\n→ Selected: {model_name}")
best_model.fit(X_all, y)

Evaluating Random Forest (5-fold CV)...
Evaluating Gradient Boosting (5-fold CV)...

Random Forest  CV-RMSE: 1.0664
Gradient Boost CV-RMSE: 1.0653

→ Selected: Gradient Boosting


GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=300,
                          random_state=42)

In [3]:
# ── Feature importance — select >5% OR top 6, whichever gives more features ──
importances = pd.Series(
    best_model.feature_importances_, index=CANDIDATE_FEATURES
).sort_values(ascending=False)

print("Feature Importances:")
for feat, imp in importances.items():
    bar = '█' * int(imp * 60)
    flag = ' ◀ selected' if imp >= 0.05 else ''
    print(f"  {feat:12s}  {imp:.3f}  {bar}{flag}")

over_threshold = importances[importances >= 0.05].index.tolist()
sig_features = over_threshold if len(over_threshold) >= 6 else importances.head(6).index.tolist()

print(f"\nFinal selection ({len(sig_features)} features): {sig_features}")

# Retrain on selected features only
best_model.fit(X_all[sig_features], y)
final_rmse = -cross_val_score(best_model, X_all[sig_features], y, cv=5,
                               scoring='neg_root_mean_squared_error').mean()
print(f"Final model CV-RMSE (selected features): {final_rmse:.4f}")

Feature Importances:
  Phaeop        0.721  ███████████████████████████████████████████ ◀ selected
  O2ml_L        0.064  ███ ◀ selected
  Salnty        0.056  ███ ◀ selected
  T_degC        0.044  ██
  PO4uM         0.035  ██
  NP_Ratio      0.026  █
  O2Sat         0.019  █
  SiO3uM        0.018  █
  NO3uM         0.011  
  Cloud_Amt     0.006  

Final selection (6 features): ['Phaeop', 'O2ml_L', 'Salnty', 'T_degC', 'PO4uM', 'NP_Ratio']
Final model CV-RMSE (selected features): 1.0553


In [ ]:
# ── Interactive heatmap ───────────────────────────────────────────────────────

# Build fine prediction grid over CA coast
LAT_MIN, LAT_MAX = 29.5, 35.5
LON_MIN, LON_MAX = -122.5, -116.5
RESOLUTION = 0.1

lat_pts = np.arange(LAT_MIN, LAT_MAX + RESOLUTION, RESOLUTION)
lon_pts = np.arange(LON_MIN, LON_MAX + RESOLUTION, RESOLUTION)
lon_grid, lat_grid = np.meshgrid(lon_pts, lat_pts)
grid_lats = lat_grid.ravel()
grid_lons = lon_grid.ravel()

feature_means = X_all[sig_features].mean()

def predict_grid(**slider_vals):
    n = len(grid_lats)
    data = {feat: np.full(n, slider_vals.get(feat, feature_means[feat]))
            for feat in sig_features}
    preds = best_model.predict(pd.DataFrame(data))
    return np.clip(preds, 0, None)

init_vals  = {feat: float(feature_means[feat]) for feat in sig_features}
init_preds = predict_grid(**init_vals)
chlora_p95 = float(np.percentile(y, 95))

# ── Plotly FigureWidget ───────────────────────────────────────────────────────
fig = go.FigureWidget(data=[go.Densitymapbox(
    lat=grid_lats,
    lon=grid_lons,
    z=init_preds,
    radius=10,
    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,1)']],
    reversescale=False,
    colorbar=dict(title='ChlorA<br>(μg/L)', thickness=16),
    zmin=0,
    zmax=chlora_p95,
    hovertemplate='Lat: %{lat:.2f}<br>Lon: %{lon:.2f}<br>Predicted ChlorA: %{z:.3f} μg/L<extra></extra>',
)])

fig.update_layout(
    mapbox_style="open-street-map",
    mapbox_center={"lat": 32.5, "lon": -119.5},
    mapbox_zoom=5.5,
    title=dict(text=f"Predicted Chlorophyll-A  ({model_name})  |  CV-RMSE: {final_rmse:.4f}",
               x=0.5, font=dict(size=14)),
    height=620,
    margin=dict(l=0, r=0, t=45, b=0),
)

# ── Sliders ───────────────────────────────────────────────────────────────────
sliders = {}
for feat in sig_features:
    col      = X_all[feat]
    q05, q95 = col.quantile(0.05), col.quantile(0.95)
    step     = (q95 - q05) / 100
    sliders[feat] = widgets.FloatSlider(
        value=round(float(feature_means[feat]), 4),
        min=round(float(q05), 4),
        max=round(float(q95), 4),
        step=round(float(step), 5),
        description=feat,
        continuous_update=False,          # fire only on mouse-release for speed
        style={'description_width': '120px'},
        layout=widgets.Layout(width='520px'),
    )

status_label = widgets.Label(value="")

def on_change(change):
    status_label.value = "Computing…"
    vals  = {feat: sliders[feat].value for feat in sig_features}
    preds = predict_grid(**vals)
    with fig.batch_update():
        fig.data[0].z = preds
    status_label.value = ""

for s in sliders.values():
    s.observe(on_change, names='value')

slider_box = widgets.VBox(
    [widgets.HTML("<b style='font-size:13px'>Adjust environmental conditions (non-slider features held at mean):</b>"),
     status_label]
    + list(sliders.values()),
    layout=widgets.Layout(padding='8px 12px')
)

display(widgets.VBox([fig, slider_box]))

    'data': [{'colorbar': {'thickness': 16, 'title': {'text': 'ChlorA<br>(μg/L)'…

ValueError: 
Invalid property path 'mapbox._derived' for layout


ValueError: 
Invalid property path 'mapbox._derived' for layout


ValueError: 
Invalid property path 'mapbox._derived' for layout


ValueError: 
Invalid property path 'mapbox._derived' for layout


ValueError: 
Invalid property path 'mapbox._derived' for layout


ValueError: 
Invalid property path 'mapbox._derived' for layout
